# Lab 1 — Can We Trust the Data?
## From a Messy CRM Export to Reusable Data Products

**Mission:** Clean a fictional recruiting-event export and produce the two artifacts used by the recommender lab:

1. `clean_recruiting_events.csv` — one validated row per engagement.
2. `school_summary.csv` — one aggregated row per school.

The raw file contains 36 schools, hundreds of legitimate repeated events, aliases, misspellings, duplicate IDs, mixed dates, missing keys, and invalid funnel values.

**Learning goals**

- Profile data before transforming it.
- Resolve school and action identities using explicit reference rules.
- Preserve legitimate repeated events while removing duplicates.
- Validate `contacts ≥ appointments ≥ qualified ≥ contracts`.
- Produce event-level and school-level artifacts with auditable lineage.

> **Completed instructor version.** Exercise values and functions are filled in, self-checks are executed, and explanations follow each solution. 

> **Use your coding assistant as a teammate.** Give it the current cell, the self-check output, and the goal. Ask it to explain the smallest useful change rather than rewriting the notebook.

Suggested prompt:

> I am working in a classroom Jupyter notebook. Explain what this self-check is testing, then suggest the smallest edit to the marked variables. Do not change the data or the test.

In [1]:
from IPython.display import display, Markdown

def check(name, condition, hint=""):
    try:
        passed = bool(condition)
    except Exception as exc:
        passed = False
        hint = f"{hint} ({type(exc).__name__}: {exc})"
    icon = "✅" if passed else "❌"
    print(f"{icon} {name}")
    if not passed and hint:
        print(f"   Hint: {hint}")
    return passed

def mission_header(text):
    display(Markdown(f"> **Mission checkpoint:** {text}"))

In [2]:
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

raw = pd.read_csv("https://usard-demo.netlify.app/data/raw_recruiting_events.csv")
print(f"Loaded {len(raw):,} raw rows from https://usard-demo.netlify.app/data/raw_recruiting_events.csv")
raw.head()

Loaded 494 raw rows from ../data/raw_recruiting_events.csv


,engagement_id,event_date,school_name,action,recruiter_hours,contacts,appointments,qualified,contracts,access_score,distance_miles
0,E1236,2026-03-03,Parkview HS,Healthcare Career Session,8,34,9,5,3.0,0.92,19
1,E1199,2026-01-21,Oak Valley High,General Recruiting Table,4,45,9,4,1.0,0.72,9
2,E1148,2025-12-23,Central High,Healthcare Careers Session,7,40,20,13,6.0,0.80,12
3,E1227,2025-12-11,CEDAR GROVE HIGH,Education Benefits Session,5,31,11,7,4.0,0.94,6
4,E1196,2025/11/12,Oak Valley HS,Benefits Session,5,36,9,6,0.0,0.72,9


## 1. Profile before fixing

Pause and predict: how many exact duplicates, missing fields, and suspicious labels do you expect?

In [3]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(dropna=True),
})
display(profile)
print("Rows:", len(raw))
print("Exact duplicate rows:", raw.duplicated().sum())
print("Distinct raw school labels:", raw["school_name"].nunique(dropna=True))
print("Distinct raw action labels:", raw["action"].nunique(dropna=True))

,dtype,missing,unique
engagement_id,object,3,479
event_date,object,1,367
school_name,object,2,135
action,object,2,18
recruiter_hours,int64,0,9
contacts,int64,0,44
appointments,int64,0,25
qualified,int64,0,15
contracts,float64,2,11
access_score,float64,0,22


Rows: 494
Exact duplicate rows: 12
Distinct raw school labels: 135
Distinct raw action labels: 18


In [4]:
check("The raw export contains 494 rows", len(raw) == 494)
check("Twelve exact duplicate rows are visible", raw.duplicated().sum() == 12)
check("Raw labels exceed the 36 real schools", raw["school_name"].nunique() > 36)

✅ The raw export contains 494 rows
✅ Twelve exact duplicate rows are visible
✅ Raw labels exceed the 36 real schools


True

## 2. Resolve school identities

Most variants can be normalized mechanically. Three genuine misspellings require explicit decisions. Complete `MANUAL_SCHOOL_FIXES`; do not use unrestricted fuzzy matching.

In [5]:
school_names = [
    "Lincoln High", "Jefferson High", "Washington High", "Roosevelt High",
    "North County Tech", "Lakeside Academy", "Madison High", "Franklin High",
    "Central High", "Riverside High", "Eastview High", "Westfield High",
    "Pine Ridge High", "Oak Valley High", "Summit High", "Cedar Grove High",
    "Parkview High", "Liberty High", "Monroe High", "Adams High",
    "Hamilton High", "Kennedy High", "Jackson High", "Grant High",
    "Wilson High", "Heritage High", "Valley Tech", "Mountain View High",
    "Harbor High", "Brookside High", "Greenfield High", "Redstone High",
    "Horizon High", "Pioneer High", "Union High", "Victory High",
]
SCHOOL_REFERENCE = {
    name.upper(): (f"S{i:03d}", name)
    for i, name in enumerate(school_names, start=1)
}

MANUAL_SCHOOL_FIXES = {
    "JEFFRSON HIGH": "JEFFERSON HIGH",
    "N COUNTY TECHNICAL": "NORTH COUNTY TECH",
    "LAKESIDE ACAD": "LAKESIDE ACADEMY",
}

def normalize_school_label(value):
    if pd.isna(value) or not str(value).strip():
        return None
    label = re.sub(r"\s+", " ", str(value).strip().replace(".", "")).upper()
    label = MANUAL_SCHOOL_FIXES.get(label, label)
    label = re.sub(r" HIGH SCHOOL$", " HIGH", label)
    label = re.sub(r" HS$", " HIGH", label)
    return label

working = raw.copy()
working["school_label"] = working["school_name"].map(normalize_school_label)
working["school_id"] = working["school_label"].map(lambda x: SCHOOL_REFERENCE.get(x, (None, None))[0])
working["school_name_clean"] = working["school_label"].map(lambda x: SCHOOL_REFERENCE.get(x, (None, None))[1])

unresolved_schools = working.loc[
    working["school_name"].notna() & working["school_id"].isna(), "school_name"
].value_counts()
unresolved_schools

Series([], Name: count, dtype: int64)

### Solution explanation — entity resolution

The three genuine misspellings map to canonical names through explicit, reviewable rules. Mechanical normalization handles capitalization, whitespace, periods, `HS`, and `High School` without unrestricted fuzzy matching.

In [6]:
check("All nonblank school labels resolve", unresolved_schools.empty,
      "Complete the three explicit mappings in MANUAL_SCHOOL_FIXES.")
check("Exactly 36 canonical schools are represented", working["school_id"].nunique() == 36)

✅ All nonblank school labels resolve
✅ Exactly 36 canonical schools are represented


True

## 3. Standardize engagement actions

Complete the alias map. Different spellings of the same action must not become different matrix columns later.

In [7]:
canonical_actions = [
    "Cyber Careers Event", "STEM Careers Presentation", "Mechanical Careers Demo",
    "Healthcare Careers Session", "Education Benefits Session", "General Recruiting Table",
]
ACTION_NAME_MAP = {action.upper(): action for action in canonical_actions}
ACTION_NAME_MAP.update({
    "CYBER CAREER EVENT": "Cyber Careers Event",
    "STEM PRESENTATION": "STEM Careers Presentation",
    "STEM CAREER PRESENTATION": "STEM Careers Presentation",
    "MECHANICAL CAREER DEMO": "Mechanical Careers Demo",
    "MECH CAREERS DEMO": "Mechanical Careers Demo",
    "HEALTHCARE CAREER SESSION": "Healthcare Careers Session",
    "HEALTH CAREERS SESSION": "Healthcare Careers Session",
    "EDUCATION BENEFIT SESSION": "Education Benefits Session",
    "BENEFITS SESSION": "Education Benefits Session",
    "GENERAL RECRUITMENT TABLE": "General Recruiting Table",
    "RECRUITING TABLE": "General Recruiting Table",
})

working["action_label"] = working["action"].map(
    lambda value: None if pd.isna(value) else str(value).strip().upper()
)
working["action_clean"] = working["action_label"].map(ACTION_NAME_MAP)

unresolved_actions = working.loc[
    working["action"].notna() & working["action_clean"].isna(), "action"
].value_counts()
unresolved_actions

Series([], Name: count, dtype: int64)

### Solution explanation — action aliases

All observed aliases map to six canonical engagement actions. This prevents spelling variants from becoming separate columns in the recommender matrix.

In [8]:
check("All nonblank action aliases resolve", unresolved_actions.empty,
      "Map singular, abbreviated, and alternate action names to the six canonical actions.")
check("Exactly six canonical actions remain", working["action_clean"].nunique() == 6)

✅ All nonblank action aliases resolve
✅ Exactly six canonical actions remain


True

## 4. Parse dates and remove duplicate records

Repeated events are legitimate. Repeated **engagement IDs** are not. Set the duplicate policy after inspecting the evidence.

In [9]:
REMOVE_DUPLICATE_IDS = True  # Keep one record per engagement ID

working["event_date_clean"] = pd.to_datetime(
    working["event_date"], errors="coerce", format="mixed"
)
duplicate_id = (
    working["engagement_id"].notna()
    & working.duplicated(subset="engagement_id", keep="first")
)
print("Duplicate engagement IDs:", duplicate_id.sum())
deduped = working.loc[~duplicate_id].copy() if REMOVE_DUPLICATE_IDS else working.copy()

Duplicate engagement IDs: 12


### Solution explanation — deduplication

Twelve repeated engagement IDs are exact duplicates, so one copy is retained. Legitimate repeated events have different IDs and remain in the history.

In [10]:
check("Duplicate engagement IDs are removed", REMOVE_DUPLICATE_IDS and deduped["engagement_id"].dropna().is_unique,
      "Set REMOVE_DUPLICATE_IDS=True; legitimate repeated events have different IDs.")
check("Four dates cannot be parsed", working["event_date_clean"].isna().sum() == 4)

✅ Duplicate engagement IDs are removed
✅ Four dates cannot be parsed


True

## 5. Validate each event

Convert numeric fields, create explicit rejection flags, and choose whether invalid rows enter the model-ready artifact.

In [11]:
numeric_fields = [
    "recruiter_hours", "contacts", "appointments", "qualified", "contracts",
    "access_score", "distance_miles",
]
for field in numeric_fields:
    deduped[field] = pd.to_numeric(deduped[field], errors="coerce")

deduped["missing_key"] = (
    deduped["engagement_id"].isna()
    | deduped["school_id"].isna()
    | deduped["action_clean"].isna()
)
deduped["invalid_date"] = deduped["event_date_clean"].isna()
deduped["missing_numeric"] = deduped[numeric_fields].isna().any(axis=1)
deduped["negative_value"] = deduped[numeric_fields].lt(0).any(axis=1)
deduped["invalid_funnel"] = ~(
    deduped["contacts"].ge(deduped["appointments"])
    & deduped["appointments"].ge(deduped["qualified"])
    & deduped["qualified"].ge(deduped["contracts"])
)

flag_columns = ["missing_key", "invalid_date", "missing_numeric", "negative_value", "invalid_funnel"]
deduped["is_valid"] = ~deduped[flag_columns].any(axis=1)
validation_summary = deduped[flag_columns + ["is_valid"]].agg(["sum"]).T
validation_summary.columns = ["row_count"]
validation_summary

,row_count
missing_key,7
invalid_date,4
missing_numeric,2
negative_value,4
invalid_funnel,10
is_valid,459


In [12]:
INVALID_ROW_POLICY = "exclude"  # Reject records that fail any validation rule

selected = deduped.loc[deduped["is_valid"]].copy() if INVALID_ROW_POLICY == "exclude" else deduped.copy()
clean_columns = [
    "engagement_id", "event_date_clean", "school_id", "school_name_clean", "action_clean",
    "recruiter_hours", "contacts", "appointments", "qualified", "contracts",
    "access_score", "distance_miles",
]
clean_events = selected[clean_columns].rename(columns={
    "event_date_clean": "event_date",
    "school_name_clean": "school_name",
    "action_clean": "action",
})

integer_columns = ["recruiter_hours", "contacts", "appointments", "qualified", "contracts", "distance_miles"]
if INVALID_ROW_POLICY == "exclude":
    clean_events[integer_columns] = clean_events[integer_columns].astype(int)
clean_events = clean_events.sort_values(["event_date", "engagement_id"]).reset_index(drop=True)
clean_events.head()

,engagement_id,event_date,school_id,school_name,action,recruiter_hours,contacts,appointments,qualified,contracts,access_score,distance_miles
0,E1247,2025-09-01,S019,Monroe High,Cyber Careers Event,9,32,19,13,8,0.61,30
1,E1280,2025-09-01,S021,Hamilton High,Healthcare Careers Session,8,26,14,9,6,0.69,33
2,E1308,2025-09-01,S023,Jackson High,General Recruiting Table,5,24,9,4,2,0.77,38
3,E1478,2025-09-01,S035,Union High,Education Benefits Session,6,26,14,6,2,0.86,23
4,E1039,2025-09-02,S002,Jefferson High,Education Benefits Session,5,30,16,7,3,0.92,12


### Solution explanation — event validation

Rows with missing keys, invalid dates, missing or negative numeric values, or impossible funnel order are excluded from the model-ready artifact. Each rule remains a separate audit flag.

In [13]:
check("Invalid records are excluded", INVALID_ROW_POLICY == "exclude")
check("Twenty-three unique records are rejected", (~deduped["is_valid"]).sum() == 23)
check("The clean event artifact contains 459 rows", len(clean_events) == 459)
check("Every clean row obeys the funnel",
      (clean_events["contacts"] >= clean_events["appointments"]).all()
      and (clean_events["appointments"] >= clean_events["qualified"]).all()
      and (clean_events["qualified"] >= clean_events["contracts"]).all())

✅ Invalid records are excluded
✅ Twenty-three unique records are rejected
✅ The clean event artifact contains 459 rows
✅ Every clean row obeys the funnel


True

## 6. Create the school summary

Aggregate all valid events by school. Rates are intentionally left for Lab 2 to calculate.

In [14]:
school_summary = (
    clean_events
    .groupby(["school_id", "school_name"], as_index=False)
    .agg(
        historical_events=("engagement_id", "count"),
        recruiter_hours=("recruiter_hours", "sum"),
        contacts=("contacts", "sum"),
        appointments=("appointments", "sum"),
        qualified=("qualified", "sum"),
        contracts=("contracts", "sum"),
        access_score=("access_score", "first"),
        distance_miles=("distance_miles", "first"),
    )
    .sort_values("school_id")
    .reset_index(drop=True)
)
school_summary.head(8)

,school_id,school_name,historical_events,recruiter_hours,contacts,appointments,qualified,contracts,access_score,distance_miles
0,S001,Lincoln High,23,147,1046,495,143,48,0.90,8
1,S002,Jefferson High,19,117,655,252,145,64,0.92,12
2,S003,Washington High,17,105,672,226,126,60,0.72,18
3,S004,Roosevelt High,12,83,469,152,81,26,0.95,6
4,S005,North County Tech,14,95,507,173,98,48,0.80,22
5,S006,Lakeside Academy,12,82,404,151,84,27,0.94,32
6,S007,Madison High,19,118,737,240,139,49,0.77,35
7,S008,Franklin High,14,91,479,148,80,23,0.71,47


In [15]:
check("The summary contains 36 schools", len(school_summary) == 36)
check("The summary has one row per school", school_summary["school_id"].is_unique)
check("Summary event counts reconcile to clean events", school_summary["historical_events"].sum() == len(clean_events))
check("No arbitrary data-quality score is present", "data_quality" not in school_summary.columns)

✅ The summary contains 36 schools
✅ The summary has one row per school
✅ Summary event counts reconcile to clean events
✅ No arbitrary data-quality score is present


True

## 7. Compare with the prepared Lab 2 artifacts

Lab 2 includes validated copies so it remains runnable even if a team does not finish Lab 1. Your results should match those prepared files.

In [16]:
expected_clean = pd.read_csv("https://usard-demo.netlify.app/data/clean_recruiting_events.csv", parse_dates=["event_date"])
expected_summary = pd.read_csv("https://usard-demo.netlify.app/data/school_summary.csv")

def frames_match(left, right):
    try:
        pd.testing.assert_frame_equal(left.reset_index(drop=True), right.reset_index(drop=True), check_dtype=False)
        return True
    except AssertionError:
        return False

check("Clean events match the prepared artifact", frames_match(clean_events, expected_clean))
check("School summary matches the prepared artifact", frames_match(school_summary, expected_summary))

✅ Clean events match the prepared artifact
✅ School summary matches the prepared artifact


True

## 8. Optional export

Leave this off during normal classroom runs. Turn it on to save your recreated artifacts separately from the prepared Lab 2 files.

In [17]:
SAVE_OUTPUTS = False
if SAVE_OUTPUTS:
    output_dir = Path("lab_outputs")
    output_dir.mkdir(exist_ok=True)
    clean_events.to_csv(output_dir / "clean_recruiting_events.csv", index=False)
    school_summary.to_csv(output_dir / "school_summary.csv", index=False)
    deduped.loc[~deduped["is_valid"]].to_csv(output_dir / "rejected_events.csv", index=False)
    print(f"Saved artifacts to {output_dir.resolve()}")

## Handoff to Lab 2

- Lab 2A loads `school_summary.csv` to rank schools.
- Lab 2B loads `clean_recruiting_events.csv` to recommend actions.

**Reflection:** Why is an explicit rejection table more useful than a single “data quality” score?